# 01 - Pré-processamento

Pré-processamento do ficheiro *accepted* (Fase 2 do enunciado, *Pre-processing*).

**Ordem dos passos**

O enunciado pede: *"Define partitions before data-dependent preprocessing"*. Por isso, antes de olhar para valores em falta, outliers ou qualidade dos dados, faz-se só o que não depende de nenhuma estatística dos dados:
1. Remover as linhas que não são empréstimos (regra estrutural).
2. Separar as partições temporais definidas na Fase 1 (`docs/01_formulacao_problema/desenho_split.md`).

A partir daqui, todas as análises e decisões de limpeza usam **só o conjunto de desenvolvimento**. O teste final fica guardado à parte e só recebe as mesmas regras, sem as rever.

**Antes de correr:** é preciso ter corrido `00_conversao_dados.ipynb`. O ficheiro *accepted* ocupa cerca de 4 GB em memória.

In [1]:
# Setup: imports e caminhos do projeto
import gc
import sys
from pathlib import Path

# Importar o pacote do projeto.
try:
    import lending
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import pandas as pd

from lending import config
from lending.cleaning import (MODIFIED_RATE, amortization_installment, find_modified_terms,
                              find_non_loan_rows)
from lending.cleaner import AcceptedCleaner
from lending.cohorts import AT_RISK_STATUSES, CohortBuilder
from lending.exclusions import ExclusionLog, write_exclusions_markdown
from lending.parsing import parse_month_year
from lending.partition import TemporalPartitioner
from lending.quality import (DISGUISED_NEVER_PAIRS, DuplicateReport, MissingnessReport,
                             QualityChecks, recode_cap_codes, recode_disguised_missing)
from lending.schema import SchemaParser, SchemaReport
from lending.storage import PickleStore

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 1. Linhas que não são empréstimos

O CSV do Kaggle é a junção de várias exportações da LendingClub. Cada exportação termina com duas linhas de totais (`Total amount funded in policy code 1/2: ...`) e há também um separador (`Loans that do not meet the credit policy`). Estas linhas estão **a meio do ficheiro**, não só no fim.

**Regra:** o `id` não é um número inteiro **e** todas as outras colunas estão vazias. Se aparecer um `id` não numérico numa linha com dados, o código dá erro em vez de a remover, porque já não seria um rodapé.

A regra é estrutural, por isso aplica-se ao ficheiro inteiro, antes da partição. Cada remoção fica no registo de exclusões, que no fim dá a tabela de exclusões do relatório.

In [2]:
accepted = PickleStore.load(config.ACCEPTED_RAW_PKL, expected_type=pd.DataFrame)
print(f"accepted: {len(accepted):,} linhas x {accepted.shape[1]} colunas")

non_loan = find_non_loan_rows(accepted)
print(f"Linhas que não são empréstimos: {non_loan.sum()}")

# O texto de cada linha marcada e quantas vezes aparece
accepted.loc[non_loan, "id"].str.replace(r"\d+$", "<valor>", regex=True).value_counts()

accepted: 2,260,701 linhas x 151 colunas
Linhas que não são empréstimos: 33


id
Total amount funded in policy code 1: <valor>    16
Total amount funded in policy code 2: <valor>    16
Loans that do not meet the credit policy          1
Name: count, dtype: int64

In [3]:
# Posição das linhas no ficheiro: confirma que estão espalhadas, entre exportações
accepted.loc[non_loan, ["id"]]

,id
421095,Total amount funded in policy code 1: 6417608175
421096,Total amount funded in policy code 2: 1944088810
528961,Total amount funded in policy code 1: 1741781700
528962,Total amount funded in policy code 2: 564202131
651664,Total amount funded in policy code 1: 1791201400
651665,Total amount funded in policy code 2: 651669342
749520,Total amount funded in policy code 1: 1443412975
749521,Total amount funded in policy code 2: 511988838
877716,Total amount funded in policy code 1: 2063142975
877717,Total amount funded in policy code 2: 823319310


In [4]:
log = ExclusionLog("accepted -> desenvolvimento")

# Só se regista: filtrar o DataFrame inteiro criaria uma cópia de ~4 GB.
# As linhas que ficam são escolhidas na partição (passo 2).
log.record(
    step="Linhas que não são empréstimos",
    kind="remoção",
    rule="id não numérico e todas as outras colunas vazias",
    reason="Totais e separadores das exportações da LendingClub, não são empréstimos",
    rows_before=len(accepted),
    rows_removed=int(non_loan.sum()),
)
log.to_frame()

,step,kind,rule,reason,rows_before,rows_removed,rows_after
n,,,,,,,
1,Linhas que não são empréstimos,remoção,id não numérico e todas as outras colunas vazias,"Totais e separadores das exportações da LendingClub, não são empréstimos",2260701,33,2260668


## 2. Partição temporal

**Data usada:** `issue_d`, o mês de financiamento (texto `Mon-AAAA`, ex.: `Dec-2015`). A conversão para data é **determinística**: uma tabela fixa de meses, sem estimar nada a partir dos dados. Por isso pode ser feita antes da partição (é aliás necessária para a fazer). Um valor fora do formato dá erro.

**Cortes** (`config.PARTITIONS`, decididos na Fase 1):

| Partição | Período (`issue_d`) | Utilização |
|---|---|---|
| Desenvolvimento | 2007-06 a 2014-12 | Treino, validação, EDA, testes de hipóteses, *fit* das transformações |
| Teste final | 2015-01 a 2015-12 | Avaliação final, uma única vez, no fim da Parte 2 |
| Reservado | 2016-01 a 2018-12 | Não usado na Parte 1 |

Os cortes são feitos ao mês, por isso os empréstimos do mesmo mês ficam sempre juntos. Cada linha tem de cair numa e uma só partição: uma data em falta ou fora dos períodos dá erro.

In [5]:
# Datas só das linhas que são empréstimos
issue_date = parse_month_year(accepted.loc[~non_loan, "issue_d"])
print(f"issue_d: {issue_date.min():%Y-%m} a {issue_date.max():%Y-%m}, "
      f"{issue_date.isna().sum()} em falta")

partitioner = TemporalPartitioner()
partitioner.summary(issue_date)

issue_d: 2007-06 a 2018-12, 0 em falta


,start,end,rows,pct
partition,,,,
development,2007-06,2014-12,466345,20.63
test,2015-01,2015-12,421095,18.63
reserved,2016-01,2018-12,1373228,60.74


In [6]:
# Empréstimos por ano de financiamento e partição
# (só contagens: não se olha para nenhum valor do teste nem do reservado)
pd.crosstab(issue_date.dt.year.rename("ano"), partitioner.assign(issue_date).rename("partição"),
            margins=True, margins_name="Total")

partição,development,test,reserved,Total
ano,,,,
2007,603,0,0,603
2008,2393,0,0,2393
2009,5281,0,0,5281
2010,12537,0,0,12537
2011,21721,0,0,21721
2012,53367,0,0,53367
2013,134814,0,0,134814
2014,235629,0,0,235629
2015,0,421095,0,421095


In [7]:
# Só se copiam para memória as partições usadas; o reservado continua no pickle original
parts = partitioner.split(accepted, issue_date, names=["development", "test"])
accepted_dev, accepted_test = parts["development"], parts["test"]

sizes = partitioner.summary(issue_date)["rows"]
assert len(accepted_dev) == sizes["development"] and len(accepted_test) == sizes["test"]

# Cada empréstimo aparece uma só vez no ficheiro: nenhum pode ficar em duas partições
assert accepted.loc[~non_loan, "id"].is_unique
print(f"ids únicos no ficheiro inteiro: {(~non_loan).sum():,} empréstimos")
# O identificador do mutuário não existe na exportação pública (usado na secção 7)
print(f"member_id preenchido no ficheiro inteiro: {accepted['member_id'].notna().sum()}")

log.record(
    step="Separar o teste final",
    kind="partição",
    rule="issue_d de 2015-01 a 2015-12",
    reason="Teste final cronológico, guardado à parte para a avaliação final (desenho_split.md)",
    rows_before=log.rows_after,
    rows_removed=int(sizes["test"]),
)
log.record(
    step="Separar o período reservado",
    kind="partição",
    rule="issue_d de 2016-01 a 2018-12",
    reason="Não usado na Parte 1, em nenhuma tarefa: é o futuro da data de treino simulada (01/01/2015) e o teste final é só 2015 (desenho_split.md)",
    rows_before=log.rows_after,
    rows_removed=int(sizes["reserved"]),
)
assert log.rows_after == len(accepted_dev)

# Libertar o ficheiro completo: a partir daqui só se usa o desenvolvimento
del accepted, parts, non_loan, issue_date
_ = gc.collect()

log.to_frame()

ids únicos no ficheiro inteiro: 2,260,668 empréstimos
member_id preenchido no ficheiro inteiro: 0


,step,kind,rule,reason,rows_before,rows_removed,rows_after
n,,,,,,,
1,Linhas que não são empréstimos,remoção,id não numérico e todas as outras colunas vazias,"Totais e separadores das exportações da LendingClub, não são empréstimos",2260701,33,2260668
2,Separar o teste final,partição,issue_d de 2015-01 a 2015-12,"Teste final cronológico, guardado à parte para a avaliação final (desenho_split.md)",2260668,421095,1839573
3,Separar o período reservado,partição,issue_d de 2016-01 a 2018-12,"Não usado na Parte 1, em nenhuma tarefa: é o futuro da data de treino simulada (01/01/2015) e o teste final é só 2015 (desenho_split.md)",1839573,1373228,466345


## 3. Guardar as partições

As partições são guardadas **ainda em texto**, sem nenhum valor alterado: este passo só separa linhas. A conversão de tipos e a limpeza vêm a seguir, sobre o desenvolvimento.

- `accepted_dev_raw.pkl`: conjunto de desenvolvimento.
- `accepted_test_raw.pkl`: teste final. Só volta a ser aberto na secção 11, para receber as mesmas regras de limpeza **sem as rever**. Não é usado na EDA nem em nenhuma decisão.
- `accepted_dev_exclusions.pkl`: o registo de exclusões, que os passos seguintes continuam.

O período reservado não é guardado: não é usado na Parte 1 e continua disponível em `accepted_raw.pkl`.

In [8]:
PickleStore.save(accepted_dev, config.ACCEPTED_DEV_RAW_PKL)
PickleStore.save(accepted_test, config.ACCEPTED_TEST_RAW_PKL)
PickleStore.save(log, config.ACCEPTED_DEV_LOG_PKL)

print(f"Desenvolvimento: {len(accepted_dev):,} linhas -> {config.ACCEPTED_DEV_RAW_PKL.name}")
print(f"Teste final:     {len(accepted_test):,} linhas -> {config.ACCEPTED_TEST_RAW_PKL.name}")
print(f"Registo:         {log} -> {config.ACCEPTED_DEV_LOG_PKL.name}")

# O teste sai da memória: a partir daqui não é usado
del accepted_test
_ = gc.collect()

Desenvolvimento: 466,345 linhas -> accepted_dev_raw.pkl
Teste final:     421,095 linhas -> accepted_test_raw.pkl
Registo:         ExclusionLog('accepted -> desenvolvimento', 3 passos, 466345 linhas) -> accepted_dev_exclusions.pkl


## 4. Conversão de tipos (parsing determinístico)

Até aqui todas as colunas são texto. Cada coluna é agora convertida para o seu tipo, segundo o schema declarado em `src/lending/schema.py`, onde o tipo de cada uma das 151 colunas está escrito à mão em vez de ser adivinhado pelo pandas.

| Tipo | Exemplo no ficheiro | Resultado | Colunas |
|---|---|---|---:|
| `id` | `"9006963"` | inteiro | 1 |
| `number` | `"10000.0"`, `"12.99"`, `"1e-10"` | decimal (`float64`) | 112 |
| `term` | `" 36 months"` | `36.0` (meses) | 1 |
| `emp_length` | `"< 1 year"`, `"10+ years"` | `0` a `10` (anos) | 1 |
| `month_year` | `"Dec-2015"` | `2015-12-01` | 11 |
| `flag` | `"Y"`/`"N"`, `"y"`/`"n"` | `True`/`False` | 3 |
| `category` | `"MORTGAGE"`, `"B3"` | texto sem espaços nas pontas | 17 |
| `text` | `"Teacher"`, url, `desc` | texto sem espaços nas pontas | 5 |

**Porque pode ser feito antes de qualquer *fit*:** cada valor é convertido só a partir de si próprio, com uma regra fixa. Nada é estimado a partir dos dados, por isso o mesmo `SchemaParser` vai converter o teste final exatamente da mesma forma, sem o rever. É o caso que o enunciado dá como exemplo: *"removing a percent sign or converting a date string using a fixed format"*.

**Garantias:**
- Um valor com formato inesperado dá **erro**, nunca passa a NaN em silêncio.
- Nenhum valor preenchido se perde. A única exceção é o texto só com espaços, que é um campo vazio e passa a valor em falta (fica contado na tabela abaixo).
- Não se mudam maiúsculas nem se juntam categorias (ex.: `"RN"` e `"Registered Nurse"`): isso é uma decisão de limpeza, não de parsing.

In [9]:
parser = SchemaParser()
accepted_dev_parsed = parser.parse(accepted_dev)
parse_summary = parser.summary(accepted_dev, accepted_dev_parsed)

print(f"{accepted_dev_parsed.shape[1]} colunas convertidas | memória: "
      f"{accepted_dev.memory_usage(deep=True).sum() / 1e6:,.0f} MB -> "
      f"{accepted_dev_parsed.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

# Quantas colunas de cada tipo e com que dtype ficaram
parse_summary.groupby(["kind", "dtype"]).size().rename("colunas").to_frame()

151 colunas convertidas | memória: 850 MB -> 667 MB


,,colunas
kind,dtype,
category,str,17
emp_length,float64,1
flag,boolean,3
id,int64,1
month_year,datetime64[ns],11
number,float64,112
term,float64,1
text,str,5


In [10]:
# Colunas em que algum valor passou a valor em falta (só pode ser texto só com espaços)
parse_summary[parse_summary["blank_to_missing"] > 0]

,kind,dtype,filled_before,filled_after,blank_to_missing
desc,text,str,125999,125764,235


In [11]:
# Antes e depois, nas mesmas linhas (amostra aleatória com semente, só para ver a conversão)
example_columns = ["id", "term", "int_rate", "emp_length", "issue_d", "earliest_cr_line",
                   "revol_util", "pymnt_plan", "home_ownership"]
example_rows = accepted_dev.sample(5, random_state=config.RANDOM_SEED).index

pd.concat({"antes (texto)": accepted_dev.loc[example_rows, example_columns],
           "depois": accepted_dev_parsed.loc[example_rows, example_columns]}, axis=1)

antes (texto)                                            \
                   id        term int_rate emp_length   issue_d   
1237956       9006963   36 months    10.15     1 year  Jul-2014   
1157714      31467963   60 months     25.8   < 1 year  Oct-2014   
1184366      28584222   36 months    12.99  10+ years  Oct-2014   
1235922      20782243   60 months    16.29        NaN  Jul-2014   
1310601      13007201   60 months    15.61  10+ years  Mar-2014   

                                                                 depois        \
        earliest_cr_line revol_util pymnt_plan home_ownership        id  term   
1237956         Apr-2000       60.4          n       MORTGAGE   9006963  36.0   
1157714         Jan-2007       61.0          n           RENT  31467963  60.0   
1184366         Sep-2003       99.6          n           RENT  28584222  36.0   
1235922         Jun-1979       72.6          n       MORTGAGE  20782243  60.0   
1310601         Jul-1999       88.7          n           RENT  13007201  60.0   

                                                                               \
        int_rate emp_length    issue_d earliest_cr_line revol_util pymnt_plan   
1237956    10.15        1.0 2014-07-01       2000-04-01       60.4      False   
1157714    25.80        0.0 2014-10-01       2007-01-01       61.0      False   
1184366    12.99       10.0 2014-10-01       2003-09-01       99.6      False   
1235922    16.29        NaN 2014-07-01       1979-06-01       72.6      False   
1310601    15.61       10.0 2014-03-01       1999-07-01       88.7      False   

                        
        home_ownership  
1237956       MORTGAGE  
1157714           RENT  
1184366           RENT  
1235922       MORTGAGE  
1310601           RENT

In [12]:
# As tabelas de conversão de term e emp_length: cada texto corresponde a um só número
# (dropna=False mostra também os valores em falta, que continuam em falta)
for col in ["term", "emp_length"]:
    mapping = (pd.DataFrame({"texto": accepted_dev[col], "valor": accepted_dev_parsed[col]})
               .value_counts(dropna=False).rename("linhas").reset_index()
               .sort_values("valor", na_position="last"))
    display(mapping.set_index("texto"))

,valor,linhas
texto,,
36 months,36.0,337996
60 months,60.0,128349


,valor,linhas
texto,,
< 1 year,0.0,36268
1 year,1.0,29622
2 years,2.0,41376
3 years,3.0,36598
4 years,4.0,28024
5 years,5.0,30779
6 years,6.0,26114
7 years,7.0,26185
8 years,8.0,22397


In [13]:
# A partir daqui trabalha-se com o desenvolvimento já convertido
# (a versão em texto continua guardada em accepted_dev_raw.pkl)
accepted_dev = accepted_dev_parsed
del accepted_dev_parsed
_ = gc.collect()

accepted_dev.dtypes.value_counts().rename("colunas").to_frame()

,colunas
float64,114
str,22
datetime64[ns],11
boolean,3
int64,1


## 5. Schema e unidades

O enunciado pede para descrever *"the source files, schema, units"*. O **significado** de cada coluna já está na tabela de elegibilidade da Fase 1 (`docs/01_formulacao_problema/elegibilidade_colunas.md`). Aqui descreve-se, para cada coluna:
- o **tipo** depois da conversão e o `dtype`;
- a **unidade** (`ACCEPTED_UNITS` em `src/lending/schema.py`): USD, %, meses, pontos FICO, contagem, etc.;
- o **intervalo observado** no desenvolvimento, que serve para confirmar a unidade. Exemplo: `int_rate` entre 5,42 e 26,06 confirma que está em percentagem (12,99 = 12,99%) e não em fração de 0 a 1.

**Convenção das percentagens:** todas as colunas em % estão na escala 0–100, não 0–1. Algumas passam de 100 (ex.: `revol_util`), o que é analisado na qualidade dos dados.

A tabela completa é escrita em `docs/02_preprocessamento/schema_accepted.md`.

In [14]:
schema_report = SchemaReport()
schema_description = schema_report.describe(accepted_dev)

# Quantas colunas há de cada unidade
schema_description["unit"].value_counts().rename("colunas").to_frame()

,colunas
unit,
contagem,42
USD,33
meses,17
categoria,17
%,11
data (mês),11
pontos FICO,6
texto,5
booleano,3


In [15]:
# Confirmação das unidades: colunas em % e pontos FICO, com o intervalo observado
schema_description[schema_description["unit"].isin(["%", "pontos FICO"])
                   & (schema_description["distinct"] > 0)]

,kind,unit,dtype,min,max,distinct,examples
column,,,,,,,
int_rate,number,%,float64,5.42,26.06,507,
dti,number,%,float64,0,39.99,3997,
fico_range_low,number,pontos FICO,float64,610,845,48,
fico_range_high,number,pontos FICO,float64,614,850,48,
revol_util,number,%,float64,0,892.30,1269,
last_fico_range_high,number,pontos FICO,float64,0,850,72,
last_fico_range_low,number,pontos FICO,float64,0,845,71,
bc_util,number,%,float64,0,339.60,1294,
pct_tl_nvr_dlq,number,%,float64,15,100,480,


In [16]:
# Datas: a última data registada em cada coluna mostra até quando o ficheiro foi atualizado
date_columns = schema_description.index[schema_description["kind"] == "month_year"]
last_dates = accepted_dev[date_columns].max().dt.strftime("%Y-%m").rename("última data")
last_dates.to_frame().assign(depois_de_2018=accepted_dev[date_columns].gt("2018-12-31").sum())

,última data,depois_de_2018
issue_d,2014-12,0
earliest_cr_line,2011-11,0
last_pymnt_d,2019-03,16157
next_pymnt_d,2019-04,12537
last_credit_pull_d,2019-03,171975
sec_app_earliest_cr_line,NaN,0
hardship_start_date,2019-03,13
hardship_end_date,2019-06,42
payment_plan_start_date,2019-04,15
debt_settlement_flag_date,2019-03,515


In [17]:
# Tabela completa do schema (151 colunas) e registo em docs/
out = schema_report.write_markdown(schema_description,
                                   config.DOCS_DIR / "02_preprocessamento" / "schema_accepted.md",
                                   dataset="accepted, desenvolvimento", n_rows=len(accepted_dev))
print("Escrito:", out.relative_to(config.PROJECT_ROOT))

with pd.option_context("display.max_rows", 200):
    display(schema_description)

Escrito: docs\02_preprocessamento\schema_accepted.md


,kind,unit,dtype,min,max,distinct,examples
column,,,,,,,
id,id,identificador,int64,,,466345,
member_id,text,texto,str,,,0,
loan_amnt,number,USD,float64,500,"35,000",1352,
funded_amnt,number,USD,float64,500,"35,000",1354,
funded_amnt_inv,number,USD,float64,0,"35,000",9843,
term,term,meses,float64,36,60,2,
int_rate,number,%,float64,5.42,26.06,507,
installment,number,USD,float64,4.93,"1,409.99",55656,
grade,category,categoria,str,,,7,"B, C, D"


## 6. Valores em falta

O enunciado pede para descrever os *"missing values"* e para *"distinguish non-applicability from missing information"*. Por isso não chega contar: é preciso saber **porque** falta cada valor, porque cada razão pede um tratamento diferente.

| Significado | Exemplo | O que quer dizer | Tratamento adequado |
|---|---|---|---|
| **sem evento** | `mths_since_last_delinq` | Nunca houve um atraso | Indicar "sem evento"; imputar a mediana inventava um atraso |
| **sem denominador** | `revol_util` | Não há limite rotativo; o rácio não existe | Indicar "não aplicável" |
| **só alguns empréstimos** | `hardship_*`, `settlement_*` | Só existe para quem teve um plano ou acordo | Pós-concessão: excluídas das tarefas |
| **só pedidos conjuntos** | `dti_joint`, `sec_app_*` | Só existe em pedidos conjuntos | Não pôr a zero (enunciado) |
| **não registado** | `emp_length` | O valor existia mas não foi guardado | Imputação aprendida no treino, com indicador |

O significado está declarado em `MISSING_MEANING` (`src/lending/quality.py`), a partir do dicionário de dados e de verificações com outras colunas. Além disso, há colunas que a LendingClub **só começou a registar mais tarde**: estão vazias nos primeiros anos por uma razão estrutural.

Esta secção só **analisa e documenta**. A imputação e os indicadores são transformações aprendidas, por isso ficam no pipeline (com *fit* só no treino). A única alteração feita aqui é a correção determinística dos códigos disfarçados (6.1).

### 6.1 Códigos disfarçados: "nunca aconteceu" guardado como 0

Em 2007–2008, `mths_since_last_record` e `mths_since_last_delinq` quase não têm valores em falta, mas têm muitos zeros. Esses zeros aparecem só em mutuários com `pub_rec` = 0 (ou `delinq_2yrs` = 0). "0 meses desde o último registo público" é impossível para quem tem 0 registos públicos, e um atraso há 0 meses contaria nos últimos 2 anos (`delinq_2yrs`). A partir de 2009 o padrão desaparece: a LendingClub mudou a forma de codificar "nunca aconteceu" de 0 para vazio.

**Regra** (determinística, dentro de cada linha): meses = 0 **e** contagem = 0 -> meses passa a valor em falta ("sem evento"). É o mesmo tipo de problema que o enunciado aponta no `Risk_Score` = 0 dos rejeitados.

In [18]:
# Por ano: % em falta e % de zeros, e quantos zeros têm a contagem também a 0
issue_year = accepted_dev["issue_d"].dt.year.rename("ano")
disguised_check = {}
for months_col, count_col in DISGUISED_NEVER_PAIRS:
    zero = accepted_dev[months_col] == 0
    disguised_check[months_col] = pd.DataFrame({
        "% em falta": accepted_dev[months_col].isna().groupby(issue_year).mean() * 100,
        "% zeros": zero.groupby(issue_year).mean() * 100,
        f"zeros com {count_col}=0": (zero & (accepted_dev[count_col] == 0)).groupby(issue_year).sum(),
        f"zeros com {count_col}>0": (zero & (accepted_dev[count_col] > 0)).groupby(issue_year).sum(),
    }).round(1)
pd.concat(disguised_check, axis=1)

mths_since_last_record                                                  \
                 % em falta % zeros zeros com pub_rec=0 zeros com pub_rec>0   
ano                                                                           
2007                    7.0    86.9                 524                   0   
2008                   60.6    31.3                 748                   0   
2009                   95.2     0.1                   3                   0   
2010                   94.0     0.0                   0                   0   
2011                   94.7     0.0                   0                   0   
2012                   97.2     0.0                   0                   0   
2013                   88.1     0.0                   0                   0   
2014                   82.4     0.0                   0                   3   

     mths_since_last_delinq                                  \
                 % em falta % zeros zeros com delinq_2yrs=0   
ano                                                           
2007                    6.5    57.2                     344   
2008                   38.1    19.8                     474   
2009                   67.0     0.0                       2   
2010                   64.3     0.0                       0   
2011                   66.2     0.0                       0   
2012                   58.8     0.0                       0   
2013                   56.5     0.1                       0   
2014                   49.2     0.1                       0   

                              
     zeros com delinq_2yrs>0  
ano                           
2007                       1  
2008                       0  
2009                       0  
2010                       0  
2011                       0  
2012                       4  
2013                     120  
2014                     278

In [19]:
accepted_dev, disguised_counts = recode_disguised_missing(accepted_dev)
disguised_counts.to_frame()

,corrigidos
mths_since_last_record,1275
mths_since_last_delinq,820


### 6.2 Resumo por coluna

- **status**: `completa` (sem valores em falta), `vazia` (100% em falta no desenvolvimento) ou `parcial`.
- **registered_from**: primeiro ano com pelo menos 1% dos valores preenchidos, ou seja, desde quando a coluna é registada. Um limiar mais alto confundiria "a coluna ainda não existia" com "o evento é raro".
- **meaning**: o significado do valor em falta (tabela acima).

In [20]:
missingness = MissingnessReport()
missing_summary = missingness.summary(accepted_dev, accepted_dev["issue_d"])

# Quantas colunas de cada estado e significado
pd.crosstab(missing_summary["meaning"].replace("", "(sem valores em falta)"),
            missing_summary["status"], margins=True, margins_name="Total")

status,completa,parcial,vazia,Total
meaning,,,,
(sem valores em falta),40,0,0,40
não registado,0,47,12,59
sem denominador,0,4,2,6
sem evento,0,8,2,10
só alguns empréstimos,0,21,0,21
só pedidos conjuntos,0,0,15,15
Total,40,80,31,151


In [21]:
# Colunas parciais, da que tem menos valores em falta para a que tem mais
with pd.option_context("display.max_rows", 200):
    display(missing_summary[missing_summary["status"] == "parcial"].sort_values("pct_missing"))

,missing,pct_missing,registered_from,status,meaning
column,,,,,
annual_inc,4,0.00,2007.0,parcial,não registado
title,20,0.00,2007.0,parcial,não registado
delinq_2yrs,29,0.01,2007.0,parcial,não registado
earliest_cr_line,29,0.01,2007.0,parcial,não registado
inq_last_6mths,29,0.01,2007.0,parcial,não registado
total_acc,29,0.01,2007.0,parcial,não registado
open_acc,29,0.01,2007.0,parcial,não registado
pub_rec,29,0.01,2007.0,parcial,não registado
acc_now_delinq,29,0.01,2007.0,parcial,não registado


### 6.3 Colunas registadas mais tarde

Muitas colunas do *bureau* só começam a ser registadas em 2012. Antes disso estão vazias para **todos** os empréstimos, por uma razão estrutural (a LendingClub ainda não as guardava) e não por causa do mutuário. A tabela mostra a % em falta por ano de concessão.

In [22]:
late_columns = missing_summary.index[(missing_summary["status"] == "parcial")
                                     & (missing_summary["registered_from"] > 2007)
                                     & (missing_summary["meaning"] != "só alguns empréstimos")]
print(f"{len(late_columns)} colunas só registadas depois de 2007, a partir de: "
      f"{missing_summary.loc[late_columns, 'registered_from'].value_counts().to_dict()}")

missing_by_year = missingness.by_year(accepted_dev, accepted_dev["issue_d"])
with pd.option_context("display.max_rows", 100):
    display(missing_by_year.loc[late_columns].round(1)
            .assign(registered_from=missing_summary.loc[late_columns, "registered_from"]))

37 colunas só registadas depois de 2007, a partir de: {2012.0: 37}


ano,2007,2008,2009,2010,2011,2012,2013,2014,registered_from
column,,,,,,,,,
mths_since_last_major_derog,100.0,100.0,100.0,100.0,100.0,89.9,79.9,71.8,2012.0
tot_coll_amt,100.0,100.0,100.0,100.0,100.0,52.0,0.0,0.0,2012.0
tot_cur_bal,100.0,100.0,100.0,100.0,100.0,52.0,0.0,0.0,2012.0
total_rev_hi_lim,100.0,100.0,100.0,100.0,100.0,52.0,0.0,0.0,2012.0
acc_open_past_24mths,100.0,100.0,100.0,100.0,100.0,14.0,0.0,0.0,2012.0
avg_cur_bal,100.0,100.0,100.0,100.0,100.0,52.0,0.0,0.0,2012.0
bc_open_to_buy,100.0,100.0,100.0,100.0,100.0,15.0,0.7,1.0,2012.0
bc_util,100.0,100.0,100.0,100.0,100.0,15.1,0.8,1.1,2012.0
mo_sin_old_il_acct,100.0,100.0,100.0,100.0,100.0,54.2,3.7,3.0,2012.0


### 6.4 Colunas com valores em falta exatamente nas mesmas linhas

Quando várias colunas faltam sempre juntas, vêm da mesma fonte (o mesmo relatório do *bureau*, o mesmo plano de *hardship*). Isto confirma as explicações acima e mostra que o tratamento deve ser feito por grupo.

In [23]:
pattern_groups = missingness.identical_patterns(accepted_dev)
pd.DataFrame({
    "colunas": [len(g) for g in pattern_groups],
    "linhas em falta": [int(accepted_dev[g[0]].isna().sum()) for g in pattern_groups],
    "grupo": [", ".join(g) for g in pattern_groups],
})

,colunas,linhas em falta,grupo
0,13,465762,"hardship_type, hardship_reason, hardship_status, deferral_term, hardship_amount, hardship_start_date, hardship_end_date, payment_plan_start_date, hardship_length, hardship_dpd, hardship_loan_status, hardship_payoff_balance_amount, hardship_last_payment_amount"
1,6,459896,"debt_settlement_flag_date, settlement_status, settlement_date, settlement_amount, settlement_percentage, settlement_term"
2,2,70277,"mo_sin_old_rev_tl_op, mo_sin_rcnt_rev_tl_op"
3,17,70276,"tot_coll_amt, tot_cur_bal, total_rev_hi_lim, mo_sin_rcnt_tl, num_accts_ever_120_pd, num_actv_bc_tl, num_actv_rev_tl, num_bc_tl, num_il_tl, num_op_rev_tl, num_rev_accts, num_rev_tl_bal_gt_0, num_tl_30dpd, num_tl_90g_dpd_24m, num_tl_op_past_12m, tot_hi_cred_lim, total_il_high_credit_limit"
4,2,58590,"num_bc_sats, num_sats"
5,4,50030,"acc_open_past_24mths, mort_acc, total_bal_ex_mort, total_bc_limit"
6,2,145,"collections_12_mths_ex_med, chargeoff_within_12_mths"
7,8,29,"delinq_2yrs, earliest_cr_line, inq_last_6mths, open_acc, pub_rec, total_acc, acc_now_delinq, delinq_amnt"


### 6.5 Consequência para as tarefas

O treino da classificação é 2007-06 a 2011-12 (desenho da Fase 1). As colunas registadas só a partir de 2012 estão **100% vazias nesse treino**, por isso o classificador não pode aprender nada com elas. Entre elas estão vários *inputs* que o enunciado sugere (ex.: `tot_cur_bal`, `bc_util`, `mort_acc`). Na regressão (treino 2007–2014) existem a partir de 2012, mas faltam nos anos anteriores.

A tabela compara a % em falta dos *inputs* sugeridos pelo enunciado no treino da classificação e nos anos mais recentes do desenvolvimento.

In [24]:
# Inputs base sugeridos pelo enunciado (secção "Note on data leakage")
suggested_inputs = [
    "emp_title", "emp_length", "home_ownership", "annual_inc", "addr_state", "zip_code", "dti",
    "fico_range_low", "fico_range_high", "earliest_cr_line", "delinq_2yrs", "inq_last_6mths",
    "open_acc", "total_acc", "pub_rec", "pub_rec_bankruptcies", "mort_acc", "revol_bal",
    "revol_util", "tot_cur_bal", "tot_hi_cred_lim", "bc_util", "bc_open_to_buy",
    "mo_sin_old_rev_tl_op", "num_actv_rev_tl", "pct_tl_nvr_dlq", "tax_liens",
]
class_train_window = accepted_dev["issue_d"] <= "2011-12-01"
recent_window = accepted_dev["issue_d"] >= "2013-01-01"

inputs_missing = pd.DataFrame({
    "% em falta 2007-2011 (treino classificação)":
        accepted_dev.loc[class_train_window, suggested_inputs].isna().mean() * 100,
    "% em falta 2013-2014": accepted_dev.loc[recent_window, suggested_inputs].isna().mean() * 100,
    "registered_from": missing_summary.loc[suggested_inputs, "registered_from"],
}).round(1).sort_values("% em falta 2007-2011 (treino classificação)", ascending=False)

unusable = inputs_missing.index[inputs_missing.iloc[:, 0] == 100]
print(f"{len(unusable)} dos {len(suggested_inputs)} inputs sugeridos estão 100% vazios no "
      f"treino da classificação: {list(unusable)}")
inputs_missing

8 dos 27 inputs sugeridos estão 100% vazios no treino da classificação: ['num_actv_rev_tl', 'pct_tl_nvr_dlq', 'mort_acc', 'tot_cur_bal', 'tot_hi_cred_lim', 'bc_util', 'bc_open_to_buy', 'mo_sin_old_rev_tl_op']


,% em falta 2007-2011 (treino classificação),% em falta 2013-2014,registered_from
num_actv_rev_tl,100.0,0.0,2012.0
pct_tl_nvr_dlq,100.0,0.0,2012.0
mort_acc,100.0,0.0,2012.0
tot_cur_bal,100.0,0.0,2012.0
tot_hi_cred_lim,100.0,0.0,2012.0
bc_util,100.0,1.0,2012.0
bc_open_to_buy,100.0,0.9,2012.0
mo_sin_old_rev_tl_op,100.0,0.0,2012.0
emp_title,6.2,5.9,2007.0
pub_rec_bankruptcies,3.2,0.0,2007.0


## 7. Duplicados e mutuários repetidos

O enunciado pede: *"Address identifiable repeated borrowers and duplicate loans (but explain limitations when identifiers are unavailable)"*. São dois problemas diferentes:

- **Empréstimo duplicado:** o mesmo empréstimo em mais do que uma linha. Inflaciona os dados e, se as cópias caírem em partições diferentes, o modelo é avaliado com casos que já viu. Verifica-se pelo `id`, pelo `url` (que contém o `id`) e pelas restantes colunas.
- **Mutuário repetido:** a mesma pessoa com vários empréstimos. Não é um erro (são empréstimos reais), mas as linhas deixam de ser independentes. O identificador do mutuário (`member_id`) está vazio em todo o ficheiro, por isso só é possível uma **estimativa** com uma chave aproximada.

Na secção 2 confirmou-se que os `id` são todos diferentes no ficheiro inteiro, por isso nenhum empréstimo pode estar no desenvolvimento e no teste ao mesmo tempo.

In [25]:
duplicates = DuplicateReport()

# Verificações de duplicados no desenvolvimento (rows = linhas que falham a verificação)
duplicates.checks(accepted_dev)

,rule,rows
check,,
id repetido,id aparece em mais do que uma linha,0
url de outro empréstimo,o loan_id do url difere do id,0
linha repetida,"todas as colunas iguais, exceto id e url",0
identificador do mutuário,member_id preenchido,0


### 7.1 Estimativa de mutuários repetidos

Sem `member_id`, usa-se uma **chave aproximada** com atributos que mudam pouco entre dois pedidos da mesma pessoa (`REPEAT_BORROWER_KEY`): mês da primeira linha de crédito (`earliest_cr_line`, fixo para cada pessoa), localização (`zip_code`, `addr_state`) e perfil declarado (`annual_inc`, `emp_title`, `home_ownership`).

**Limitações da estimativa:**
- **Falsos negativos:** se a pessoa mudar de emprego, de rendimento ou de casa entre os dois pedidos, a chave muda e a repetição não é detetada.
- **Falsos positivos:** duas pessoas diferentes podem coincidir em todas as colunas (o `zip_code` só tem 3 dígitos).
- Linhas com alguma coluna da chave vazia (sobretudo `emp_title`) não entram.

Por isso o resultado é uma **ordem de grandeza**, não uma lista de mutuários.

In [26]:
repeat_candidates = duplicates.repeat_borrowers(accepted_dev)
display(duplicates.repeat_summary(accepted_dev).to_frame("valor"))

# Empréstimos por grupo, e grupos com empréstimos no treino da classificação (até 2011)
# e depois dele: a mesma pessoa pode aparecer em lados diferentes de um corte temporal
group_dates = repeat_candidates.groupby("group")["issue_d"]
crosses_2012 = ((group_dates.min() <= "2011-12-01") & (group_dates.max() >= "2012-01-01")).sum()
print("empréstimos por grupo:", repeat_candidates.groupby("group").size().value_counts().sort_index().to_dict())
print(f"grupos com empréstimos até 2011 e a partir de 2012: {crosses_2012}")

# Exemplos (os primeiros grupos, ordenados pela chave)
repeat_candidates.head(8)

,valor
empréstimos com chave completa,438729
empréstimos em grupos repetidos,2106
% dos empréstimos com chave,0.48
grupos (possíveis mutuários),1049
intervalo mediano no grupo (meses),8.0
grupos com todos os empréstimos no mesmo mês,39


empréstimos por grupo: {2: 1041, 3: 8}
grupos com empréstimos até 2011 e a partir de 2012: 128


,id,issue_d,earliest_cr_line,zip_code,addr_state,annual_inc,emp_title,home_ownership,group,group_size
1902757,1579341,2012-10-01,1960-10-01,450xx,OH,170000.0,Duke Energy,MORTGAGE,0,2
1853235,4556393,2013-05-01,1960-10-01,450xx,OH,170000.0,Duke Energy,MORTGAGE,0,2
1323338,12395685,2014-02-01,1967-08-01,959xx,CA,50000.0,Paralegal,RENT,1,2
1199335,26248841,2014-09-01,1967-08-01,959xx,CA,50000.0,Paralegal,RENT,1,2
1624806,790337,2011-06-01,1971-11-01,913xx,CA,39000.0,NCO Financial,RENT,2,2
1936746,1114200,2012-01-01,1971-11-01,913xx,CA,39000.0,NCO Financial,RENT,2,2
1348078,10143356,2014-01-01,1972-02-01,921xx,CA,70000.0,Office Manager,RENT,3,2
1189170,28262417,2014-10-01,1972-02-01,921xx,CA,70000.0,Office Manager,RENT,3,2


## 8. Valores suspeitos e consistência entre colunas

O enunciado pede *"quality checks"* e dá como exemplo um artefacto de introdução de dados (DTI de 4042.11% nos rejeitados). Fazem-se dois tipos de verificação (`QualityChecks` em `src/lending/quality.py`):

- **Intervalo:** valores fora do esperado numa coluna (ex.: `revol_util` acima de 100%, rendimentos extremos, o valor 9 999 999).
- **Consistência:** regras que têm de valer em todas as linhas (ex.: `funded_amnt` ≤ `loan_amnt`; a prestação tem de corresponder à fórmula de amortização).

**Como se decide o que fazer:**
- Um valor que é claramente um código ou uma linha cujas condições não são as originais corrige-se ou exclui-se com uma **regra fixa** (determinística, igual no teste).
- Um valor extremo mas possível **fica**. Cortá-lo num percentil ou transformá-lo (ex.: logaritmo) é uma transformação aprendida, por isso é feita no pipeline, com *fit* só no treino.
- Pequenas incoerências entre colunas do *bureau* ficam documentadas: sem saber qual das colunas está certa, qualquer correção seria arbitrária.

In [27]:
quality_checks = QualityChecks()

# rows = linhas do desenvolvimento que falham cada regra
quality_summary = quality_checks.summary(accepted_dev)
quality_summary

,category,rule,rows,pct
check,,,,
revol_util acima de 100%,intervalo,revol_util > 100,1292,0.277
bc_util acima de 100%,intervalo,bc_util > 100,8082,1.733
limite de crédito no teto,intervalo,tot_hi_cred_lim ou total_rev_hi_lim = 9 999 999,3,0.001
rendimento acima de 1 milhão,intervalo,annual_inc > 1 000 000,59,0.013
rendimento abaixo de 4 000,intervalo,annual_inc < 4 000,6,0.001
DTI igual a 0,intervalo,dti = 0,363,0.078
FICO fora do intervalo de 4 pontos,intervalo,"fico_range_high - fico_range_low != 4, exceto o escalão 845-850",0,0.000
financiado acima do pedido,consistência,funded_amnt > loan_amnt,0,0.000
investidores acima do financiado,consistência,funded_amnt_inv > funded_amnt,0,0.000


### 8.1 Empréstimos com condições modificadas depois da concessão

Duas regras de consistência apontam para empréstimos cuja **taxa ou prestação foram alteradas depois da concessão** (ex.: renegociação):

1. **`int_rate` = 6,00% num grau B a G.** Sem estes casos, a taxa mínima de cada grau no desenvolvimento é muito superior (B 8,67%, …, G 16,59%), por isso 6,00% não é uma taxa da tabela desses graus. No grau A (mínimo 5,42%) é uma taxa possível e não é marcada.
2. **Prestação diferente da fórmula de amortização** de `funded_amnt`, `int_rate` e `term` em mais de 1 USD. Nos restantes empréstimos a diferença é de cêntimos (arredondamento).

O enunciado pede para usar os valores originais de `int_rate` e `installment` (*"not later modifications"*). Nestes empréstimos não é possível recuperar os originais, e `int_rate` é o **alvo da regressão**. Por isso são **excluídos** (registado no log de exclusões). A regra é fixa e usa só a própria linha, por isso aplica-se igual ao teste.

In [28]:
# Evidência da regra 1: taxa mínima de cada grau sem os 6,00%, e quantos 6,00% há em cada grau
is_modified_rate = accepted_dev["int_rate"] == MODIFIED_RATE
display(pd.DataFrame({
    "taxa mínima (sem 6,00%)": accepted_dev.loc[~is_modified_rate].groupby("grade")["int_rate"].min(),
    "empréstimos com 6,00%": accepted_dev.loc[is_modified_rate, "grade"].value_counts(),
}).fillna(0))

# Evidência da regra 2: diferença entre a prestação registada e a da fórmula
expected_installment = amortization_installment(accepted_dev["funded_amnt"], accepted_dev["int_rate"],
                                                accepted_dev["term"])
installment_gap = (accepted_dev["installment"] - expected_installment).abs()
print("diferença |prestação - fórmula| (USD), quantis:",
      installment_gap.quantile([0.5, 0.99, 0.999, 1]).round(3).to_dict())

,"taxa mínima (sem 6,00%)","empréstimos com 6,00%"
grade,,
A,5.42,6
B,8.67,32
C,10.28,38
D,11.86,37
E,13.43,26
F,15.01,4
G,16.59,2


diferença |prestação - fórmula| (USD), quantis: {0.5: 0.005, 0.99: 0.01, 0.999: 0.042, 1.0: 496.058}


In [29]:
modified_terms = find_modified_terms(accepted_dev)
print("empréstimos marcados:", modified_terms.sum().to_dict())
display(accepted_dev.loc[modified_terms["modified"], "loan_status"].value_counts().rename("empréstimos").to_frame())

# A exclusão é feita pela AcceptedCleaner, a mesma classe que limpa o teste na secção 11
accepted_dev = AcceptedCleaner().exclude_modified(accepted_dev, log)
log.to_frame()

empréstimos marcados: {'rate_below_grade': 139, 'installment_mismatch': 203, 'modified': 221}


,empréstimos
loan_status,
Fully Paid,196
Current,14
Charged Off,9
Late (16-30 days),1
Does not meet the credit policy. Status:Fully Paid,1


,step,kind,rule,reason,rows_before,rows_removed,rows_after
n,,,,,,,
1,Linhas que não são empréstimos,remoção,id não numérico e todas as outras colunas vazias,"Totais e separadores das exportações da LendingClub, não são empréstimos",2260701,33,2260668
2,Separar o teste final,partição,issue_d de 2015-01 a 2015-12,"Teste final cronológico, guardado à parte para a avaliação final (desenho_split.md)",2260668,421095,1839573
3,Separar o período reservado,partição,issue_d de 2016-01 a 2018-12,"Não usado na Parte 1, em nenhuma tarefa: é o futuro da data de treino simulada (01/01/2015) e o teste final é só 2015 (desenho_split.md)",1839573,1373228,466345
4,Condições modificadas depois da concessão,remoção,"int_rate = 6.00 num grau B a G, ou |installment - fórmula de amortização| > 1 USD",Taxa ou prestação alteradas depois da concessão; o enunciado pede os valores originais,466345,221,466124


### 8.2 Limites de crédito no teto (9 999 999)

`tot_hi_cred_lim` e `total_rev_hi_lim` têm exatamente 9 999 999 em 3 linhas: o maior valor que cabe em 7 dígitos, usado como teto. Numa delas o limite total é 9 999 999 mas o limite rotativo é 63 400 e o saldo total 268 743, o que mostra que não é um limite real. **Regra:** 9 999 999 exato -> valor em falta (o tratamento dos valores em falta é feito no pipeline).

In [30]:
capped_rows = (accepted_dev["tot_hi_cred_lim"] == 9_999_999) | (accepted_dev["total_rev_hi_lim"] == 9_999_999)
display(accepted_dev.loc[capped_rows, ["tot_hi_cred_lim", "total_rev_hi_lim", "tot_cur_bal",
                                       "revol_bal", "annual_inc"]])

accepted_dev, capped_counts = recode_cap_codes(accepted_dev)
capped_counts.to_frame()

,tot_hi_cred_lim,total_rev_hi_lim,tot_cur_bal,revol_bal,annual_inc
1187767,9999999.0,9999999.0,3840795.0,2560703.0,999999.0
1194988,9999999.0,63400.0,268743.0,23270.0,80000.0
1838679,9999999.0,9999999.0,3881449.0,2568995.0,1000000.0


,corrigidos
tot_hi_cred_lim,3
total_rev_hi_lim,2


### 8.3 Restantes verificações: o que fica e porquê

| Verificação | Linhas | Decisão |
|---|---:|---|
| `revol_util` > 100% | 1 292 | Fica. É possível (saldo acima do limite) e coerente com `revol_bal` e o limite: o caso de 892% tem saldo de 2 677 USD e limite de 300 USD. Valores extremos tratados no pipeline |
| `bc_util` > 100% | 8 082 | Fica, pela mesma razão |
| `annual_inc` > 1 milhão | 59 | Fica. Alguns são pouco plausíveis (ex.: 7,4 milhões com o cargo "Registered Nurse"), mas o DTI foi calculado com esse rendimento e não há forma de saber o valor certo. A assimetria do rendimento é tratada no pipeline (ex.: logaritmo) |
| `annual_inc` < 4 000 | 6 | Fica. Valores baixos mas possíveis (ex.: "disabled, student") |
| `dti` = 0 | 363 | Fica. Possível (sem outras dívidas) |
| FICO +5 pontos | 0 | Os 34 casos com intervalo de 5 pontos são todos o escalão 845–850 (o FICO acaba em 850): não são erros |
| Incoerências entre colunas do *bureau* (atrasos, saldos, `revol_util` = 0 com saldo) | 2 a 419 por regra | Ficam, documentadas. Vêm de agregados calculados em momentos ou fontes diferentes; sem saber qual coluna está certa, corrigir seria arbitrário |
| Ativo depois do fim do prazo | 88 | Fica. São empréstimos de 60 meses de 2013–2014 ainda em pagamento (prolongamentos); a classificação só usa empréstimos de 36 meses |
| Regras sempre cumpridas (`funded_amnt` ≤ `loan_amnt`, `grade` = letra da `sub_grade`, `earliest_cr_line` < `issue_d`, …) | 0 | Confirmam a coerência do ficheiro |

## 9. Coortes de cada tarefa

A limpeza (secções 1 a 8) é comum a todas as tarefas. Cada tarefa usa depois a sua **coorte**, definida na Fase 1 (`docs/01_formulacao_problema/coorte_alvo.md` e `desenho_split.md`) e ajustada com as duas decisões do grupo tomadas na limpeza (secções 9.1 e 9.2). As regras estão em `CohortBuilder` (`src/lending/cohorts.py`) e são determinísticas: o mesmo código constrói as coortes do teste no fim.

| Tarefa | Coorte no desenvolvimento |
|---|---|
| Regressão | Empréstimos de 2013-01 a 2014-12 com `int_rate` (36 e 60 meses, incluindo `Current`) |
| Classificação | 36 meses, treino 2007-06 a 2011-12 (2012–2014 é embargo), sem `Current` |
| Clustering de mutuários | Todo o conjunto limpo |

O **alvo da classificação** fica na coluna `target` (`config.CLASS_COL`), com as classes *Good standing* = Fully Paid e *Default* = Charged Off, Default, por esta ordem (`config.CLASS_ORDER`). Os "Does not meet the credit policy" entram pelo desfecho (Fully Paid ou Charged Off), e essa indicação nunca é usada como preditor.

Cada tarefa tem o seu registo de exclusões, que continua o registo comum (`branch`).

In [31]:
cohort_builder = CohortBuilder()
cohort_logs = {
    "regression": log.branch("Regressão (treino)"),
    "classification": log.branch("Classificação (treino)"),
    "clustering": log.branch("Clustering de mutuários"),
}

regression_train = cohort_builder.regression(accepted_dev, cohort_logs["regression"])
classification_train = cohort_builder.classification(accepted_dev, cohort_logs["classification"])
clustering_cohort = cohort_builder.clustering(accepted_dev, cohort_logs["clustering"])

cohort_sizes = pd.Series({
    "Regressão (treino)": len(regression_train),
    "Classificação (treino)": len(classification_train),
    "Clustering de mutuários": len(clustering_cohort),
}, name="empréstimos")

# A coorte do clustering é o conjunto limpo inteiro: não é preciso guardá-la à parte
del clustering_cohort
_ = gc.collect()

cohort_sizes.to_frame()

,empréstimos
Regressão (treino),370278
Classificação (treino),31522
Clustering de mutuários,466124


In [32]:
# Pendentes da Fase 1: "Does not meet the credit policy" e loan_status em falta no desenvolvimento
does_not_meet = accepted_dev["loan_status"].str.startswith("Does not meet")
print(f"Does not meet no desenvolvimento: {does_not_meet.sum():,} | "
      f"na coorte da classificação: {classification_train['loan_status'].str.startswith('Does not meet').sum():,}")
print(f"loan_status em falta no desenvolvimento: {accepted_dev['loan_status'].isna().sum()}")

# Alvo da classificação no treino: loan_status de origem e classe
target_table = (classification_train.groupby([config.CLASS_COL, "loan_status"], observed=False)
                .size().rename("empréstimos").reset_index())
target_table = target_table[target_table["empréstimos"] > 0]
display(target_table)

target_counts = classification_train[config.CLASS_COL].value_counts().reindex(config.CLASS_ORDER)
pd.DataFrame({"empréstimos": target_counts,
              "%": (100 * target_counts / target_counts.sum()).round(2)})

Does not meet no desenvolvimento: 2,748 | na coorte da classificação: 2,437
loan_status em falta no desenvolvimento: 0


,target,loan_status,empréstimos
2,Good standing,Does not meet the credit policy. Status:Fully Paid,1788
3,Good standing,Fully Paid,25858
4,Default,Charged Off,3227
5,Default,Does not meet the credit policy. Status:Charged Off,649


,empréstimos,%
target,,
Good standing,27646,87.7
Default,3876,12.3


### 9.1 Classificação binária: *Good standing* vs *Default*

*At risk* (em atraso ou em *grace period*) é um estado que só existe enquanto o empréstimo está **ativo**. O desenho da Fase 1 só usa empréstimos de 36 meses com o **prazo terminado** (regra de observabilidade do enunciado), e esses já fecharam todos: foram pagos ou perdidos. Por isso o treino tem **0** empréstimos *At risk*.

A célula seguinte mostra onde estão os *At risk* do desenvolvimento: todos são empréstimos de 60 meses de 2013–2014, ainda ativos na data de extração.

É um conflito entre dois pedidos do enunciado (3 classes e rótulos observáveis na data de treino simulada) que o ficheiro, com uma única fotografia do estado, não permite conciliar. **Decisão do grupo:** manter a regra de observabilidade e fazer uma classificação **binária**. No treino não há *At risk*; os que existam no teste contam como *Default* (já passaram o fim do prazo e continuam em atraso, e retirá-los seria escolher as linhas de avaliação pelo alvo). Justificação completa em `docs/02_preprocessamento/coortes.md`.

In [33]:
# Onde estão os empréstimos "At risk" no desenvolvimento (prazo e ano de concessão)
at_risk = accepted_dev["loan_status"].isin(AT_RISK_STATUSES)
pd.crosstab([accepted_dev.loc[at_risk, "term"].rename("prazo"),
             accepted_dev.loc[at_risk, "issue_d"].dt.year.rename("ano")],
            accepted_dev.loc[at_risk, "loan_status"], margins=True, margins_name="Total")

loan_status  In Grace Period  Late (16-30 days)  Late (31-120 days)  Total
prazo ano                                                                 
60.0  2013                 1                  0                   3      4
      2014               200                 72                 334    606
Total                    201                 72                 337    610

### 9.2 Regressão: treino só com 2013–2014

As 37 colunas do *bureau* registadas só a partir de 2012 (secção 6.3) estão vazias em 2007–2011 e em parte de 2012. Com o treino em 2007–2014, faltariam em cerca de 15% das linhas, e um indicador de "em falta" nessas colunas seria, na prática, um indicador do ano. **Decisão do grupo:** a regressão treina só com os empréstimos de **2013-01 a 2014-12** (`config.REGRESSION_PARTITIONS`), o primeiro período em que essas colunas estão completas. O teste continua a ser 2015.

A célula seguinte confirma a escolha, com a % em falta por ano em duas medidas: a mediana das 37 colunas e a pior das 8 que o enunciado sugere como *inputs*. Não se usa a pior das 37, porque algumas (`mths_since_recent_*`) faltam naturalmente quando o evento nunca aconteceu (secção 6).

In [34]:
# % em falta por ano de concessão nas 37 colunas registadas mais tarde (secção 6.3)
year = accepted_dev["issue_d"].dt.year
late_inputs = [c for c in suggested_inputs if c in late_columns]   # os 8 inputs sugeridos
pd.DataFrame({
    "empréstimos": year.value_counts().sort_index(),
    "mediana % em falta (37 colunas)":
        (accepted_dev[late_columns].isna().groupby(year).mean() * 100).median(axis=1).round(2),
    f"pior % em falta ({len(late_inputs)} inputs sugeridos)":
        (accepted_dev[late_inputs].isna().groupby(year).mean() * 100).max(axis=1).round(2),
    "na regressão": regression_train["issue_d"].dt.year.value_counts().sort_index(),
}).fillna(0).astype({"na regressão": int})

,empréstimos,mediana % em falta (37 colunas),pior % em falta (8 inputs sugeridos),na regressão
issue_d,,,,
2007,603,100.00,100.00,0
2008,2388,100.00,100.00,0
2009,5280,100.00,100.00,0
2010,12532,100.00,100.00,0
2011,21709,100.00,100.00,0
2012,53334,51.98,51.99,0
2013,134767,0.00,0.79,134767
2014,235511,0.00,1.11,235511


## 10. Guardar os conjuntos limpos

Até aqui, as alterações das secções 4 a 8 só existiam em memória. Guarda-se agora, com os nomes combinados no grupo (`config.py`):

| Ficheiro (`data/processed/`) | Conteúdo | Para quê |
|---|---|---|
| `aceites_dev.pkl` | Desenvolvimento limpo, 2007-06 a 2014-12 | EDA geral, clustering, testes de hipóteses aceites vs rejeitados |
| `regressao_dev.pkl` | Coorte da regressão, 2013-01 a 2014-12 | EDA e modelação da regressão |
| `classificacao_treino.pkl` | Coorte da classificação, 36 meses, 2007-06 a 2011-12, com a coluna `target` | EDA da relação com o alvo, hipóteses sobre o *default*, modelação |
| `accepted_dev_exclusions.pkl` | Registo comum, com o passo da secção 8 | Rastreabilidade |
| `accepted_dev_cohort_logs.pkl` | Registos de cada tarefa | Rastreabilidade |

Os três conjuntos mantêm as 151 colunas: a escolha das *features* de cada tarefa (tabela de elegibilidade) é feita depois, no pipeline. O teste final é preparado na secção 11 e a **tabela de exclusões** é gerada na secção 12. Os rejeitados (`rejeitados_dev.pkl`) são preparados num notebook próprio.

In [35]:
PickleStore.save(accepted_dev, config.ACCEPTED_DEV_PKL)
PickleStore.save(regression_train, config.REG_DEV_PKL)
PickleStore.save(classification_train, config.CLS_TRAIN_PKL)
PickleStore.save(log, config.ACCEPTED_DEV_LOG_PKL)
PickleStore.save(cohort_logs, config.COHORT_LOGS_PKL)

for name, data, file in [("Desenvolvimento limpo", accepted_dev, config.ACCEPTED_DEV_PKL),
                         ("Regressão (treino)", regression_train, config.REG_DEV_PKL),
                         ("Classificação (treino)", classification_train, config.CLS_TRAIN_PKL)]:
    print(f"{name}: {len(data):,} linhas x {data.shape[1]} colunas -> {file.name}")
print(f"Registos: {log} | {list(cohort_logs.values())}")

# As coortes já estão guardadas: libertar memória para a secção 11
del regression_train, classification_train
_ = gc.collect()

Desenvolvimento limpo: 466,124 linhas x 151 colunas -> aceites_dev.pkl
Regressão (treino): 370,278 linhas x 151 colunas -> regressao_dev.pkl
Classificação (treino): 31,522 linhas x 152 colunas -> classificacao_treino.pkl
Registos: ExclusionLog('accepted -> desenvolvimento', 4 passos, 466124 linhas) | [ExclusionLog('Regressão (treino)', 6 passos, 370278 linhas), ExclusionLog('Classificação (treino)', 7 passos, 31522 linhas), ExclusionLog('Clustering de mutuários', 5 passos, 466124 linhas)]


## 11. Teste final: as mesmas regras, sem as rever

O teste final (2015) foi separado na secção 2 e ficou em texto. Recebe agora **exatamente as mesmas regras** de limpeza do desenvolvimento, pela mesma ordem, sem nenhum ajuste:

1. conversão de tipos (secção 4);
2. zeros que significam "nunca aconteceu" passam a valor em falta (6.1);
3. exclusão dos empréstimos com condições modificadas (8.1);
4. códigos de teto 9 999 999 passam a valor em falta (8.2).

As regras estão juntas na classe `AcceptedCleaner` (`src/lending/cleaner.py`). São todas regras fixas, aplicadas linha a linha, que não aprendem nada com os dados. Por isso aplicá-las ao teste não é *leakage*: o que o enunciado proíbe é aprender transformações com o teste (*fit*) ou mudar decisões olhando para ele.

**O que não se faz com o teste:** não se mostra nenhuma estatística, distribuição ou exemplo. Só se registam quantas linhas cada regra removeu e quantos valores corrigiu, porque o enunciado pede que todas as exclusões fiquem documentadas. As coortes do teste (regressão e classificação) só são construídas na avaliação final, com o mesmo `CohortBuilder`.

Antes de a usar no teste, prova-se que a classe faz exatamente o que as secções 4 a 8 fizeram: aplicada ao desenvolvimento em texto (`accepted_dev_raw.pkl`), tem de dar o conjunto limpo, célula a célula.

In [36]:
# Prova: a AcceptedCleaner aplicada ao desenvolvimento em texto reproduz o conjunto limpo
cleaner = AcceptedCleaner()
check_log = ExclusionLog("verificação da AcceptedCleaner")
dev_again, dev_corrections = cleaner.clean(
    PickleStore.load(config.ACCEPTED_DEV_RAW_PKL, expected_type=pd.DataFrame), check_log)

pd.testing.assert_frame_equal(dev_again, accepted_dev)                    # mesmos valores e tipos
assert check_log.steps[-1].rows_removed == log.steps[-1].rows_removed     # mesma exclusão
assert dev_corrections["mths_since_last_record"] == disguised_counts["mths_since_last_record"]
assert dev_corrections["tot_hi_cred_lim"] == capped_counts["tot_hi_cred_lim"]
print(f"Igual ao conjunto limpo: {dev_again.shape[0]:,} x {dev_again.shape[1]} | "
      f"excluídos: {check_log.steps[-1].rows_removed} | correções: {dev_corrections.to_dict()}")

del dev_again
_ = gc.collect()

Igual ao conjunto limpo: 466,124 x 151 | excluídos: 221 | correções: {'mths_since_last_record': 1275, 'mths_since_last_delinq': 820, 'tot_hi_cred_lim': 3, 'total_rev_hi_lim': 2}


In [37]:
# Registo do teste: o mesmo início do registo comum, depois a separação e a limpeza
test_log = ExclusionLog("Teste final (2015): ficheiro accepted -> teste limpo")
first = log.steps[0]   # linhas que não são empréstimos (secção 1)
test_log.record(first.step, first.kind, first.rule, first.reason,
                rows_before=first.rows_before, rows_removed=first.rows_removed)

accepted_test_raw = PickleStore.load(config.ACCEPTED_TEST_RAW_PKL, expected_type=pd.DataFrame)
assert len(accepted_test_raw) == log.steps[1].rows_removed   # as linhas separadas na secção 2
test_log.record(
    step="Separar o desenvolvimento e o período reservado",
    kind="partição",
    rule="issue_d fora de 2015-01 a 2015-12",
    reason="Só os empréstimos do teste final (desenho_split.md)",
    rows_before=test_log.rows_after,
    rows_removed=test_log.rows_after - len(accepted_test_raw),
)

accepted_test, test_corrections = cleaner.clean(accepted_test_raw, test_log)
del accepted_test_raw
_ = gc.collect()

PickleStore.save(accepted_test, config.TEST_PKL)
PickleStore.save(test_log, config.TEST_LOG_PKL)
print(f"Teste final limpo: {len(accepted_test):,} linhas x {accepted_test.shape[1]} colunas "
      f"-> {config.TEST_PKL.name}")
print("Valores corrigidos:", test_corrections.to_dict())

del accepted_test
_ = gc.collect()
test_log.to_frame()

Teste final limpo: 420,888 linhas x 151 colunas -> teste_2015.pkl
Valores corrigidos: {'mths_since_last_record': 0, 'mths_since_last_delinq': 1, 'tot_hi_cred_lim': 3, 'total_rev_hi_lim': 1}


,step,kind,rule,reason,rows_before,rows_removed,rows_after
n,,,,,,,
1,Linhas que não são empréstimos,remoção,id não numérico e todas as outras colunas vazias,"Totais e separadores das exportações da LendingClub, não são empréstimos",2260701,33,2260668
2,Separar o desenvolvimento e o período reservado,partição,issue_d fora de 2015-01 a 2015-12,Só os empréstimos do teste final (desenho_split.md),2260668,1839573,421095
3,Condições modificadas depois da concessão,remoção,"int_rate = 6.00 num grau B a G, ou |installment - fórmula de amortização| > 1 USD",Taxa ou prestação alteradas depois da concessão; o enunciado pede os valores originais,421095,207,420888


## 12. Tabela de exclusões

Gerada a partir dos registos, para não haver números escritos à mão: o registo comum (ficheiro -> desenvolvimento limpo), o de cada tarefa e o do teste final. Fica em `docs/02_preprocessamento/tabela_exclusoes.md`.

In [38]:
out = write_exclusions_markdown(
    [log.branch("Comum: ficheiro accepted -> desenvolvimento limpo"), *cohort_logs.values(), test_log],
    config.DOCS_DIR / "02_preprocessamento" / "tabela_exclusoes.md",
)
print("Escrito:", out.relative_to(config.PROJECT_ROOT))

Escrito: docs\02_preprocessamento\tabela_exclusoes.md
